# SK-game vs quantum annealing — `main_simulation`

For every **regime** (a point in parameter space chosen so that the planner's optimum is easy or hard to retrieve) and for every **N** on a linear grid, on the *same* instances $J$:

* the **reference optimum** $R^\star=\max_S \frac1N S^\top J S$ (exact for $N\le n_{\rm exact}$, best known above);
* **classical optimisers** (greedy, SA, PT, τ-EO, SDP+GW, dimod SA/tabu);
* the **quantum annealer** via dimod (`DWaveCliqueSampler` on the QPU, or simulated annealing as a stand-in);
* the **game** itself (learning map at the regime's $\alpha,\beta,\varepsilon$): which phase it reaches and whether it lands on $R^\star$.

Why the regimes differ:

* static difficulty depends only on $J_0^{\rm eff}=J_0/(1-\varepsilon/2)$: spin glass for $J_0^{\rm eff}<1$ (hard), ferromagnet for $J_0^{\rm eff}>1$ (easy);
* the game's phase depends on $(\alpha,\beta,\varepsilon)$: fixed points (satisficing) for small $\alpha$ and $\varepsilon<\varepsilon_c\approx0.8$, 2-cycles at $\alpha=1$, chaos for $\varepsilon>\varepsilon_c$, paramagnet for $T>T_c(\varepsilon)$.

This notebook is a thin front-end to `main_simulation.py`: every stage is resumable, so you can re-run it after a job has filled part of the results. On Cineca, see `README_CINECA.md` (headless execution: `cineca/job_notebook.sbatch`).

In [ ]:
# Parameters: read from the environment so a batch job can set them without editing the notebook
import os
from pathlib import Path

OUT = Path(os.environ.get("SKGAME_OUT", "results/notebook_smoke"))   # campaign directory
PRESET = os.environ.get("SKGAME_PRESET", "smoke")                    # smoke | small | full (used only if OUT is new)
SAMPLER = os.environ.get("SKGAME_SAMPLER", "sim")                    # sim (no token) | qpu (Leap token + internet) | none
RUN_CLASSICAL = os.environ.get("SKGAME_RUN_CLASSICAL", "1") == "1"   # 0: only analyse what jobs already produced
MAX_CALLS = int(os.environ["SKGAME_MAX_CALLS"]) if "SKGAME_MAX_CALLS" in os.environ else None
print(OUT, PRESET, SAMPLER, RUN_CLASSICAL, MAX_CALLS)

In [ ]:
import main_simulation as ms

if not (OUT / "config.json").exists():
    ms.save_config(OUT, ms.make_config(PRESET))
cfg = ms.load_config(OUT)
ms.describe_plan(OUT, cfg)

## 1. References, classical solvers and the game  (compute node)

In [ ]:
if RUN_CLASSICAL:
    ms.stage_classical(OUT, cfg)

## 2. Annealer  (needs outbound HTTPS and a Leap token when `SAMPLER = "qpu"`)

The QPU back-ends of the config (default `qpu:zephyr:clique` and `qpu:pegasus:clique`) solve the same instances with the same settings; the analysis says which topology wins. With `qpu`, run `python main_simulation.py qpu --check --out ...` first: it prints each solver, its largest clique and the estimated QPU time without spending any. Embeddings and chains: `../visualization/embedding_explanation/`.

In [ ]:
# sim -> sim:logical (+ any sim:* back-end of the config); qpu -> every qpu:* back-end (Zephyr, Pegasus)
if SAMPLER == "sim":
    specs = ["sim:logical"] + [b for b in cfg.backends if b.startswith("sim:")]
elif SAMPLER == "qpu":
    specs = [b for b in cfg.backends if b.startswith("qpu:")]
else:
    specs = []
for spec in specs:
    ms.stage_anneal(OUT, cfg, spec, max_calls=MAX_CALLS)

## 3. Analysis

In [ ]:
from IPython.display import Image, display

rows = ms.stage_analyze(OUT, cfg)
for f in sorted((OUT / "summary").glob("*.png")):
    print(f.name)
    display(Image(filename=str(f)))

In [ ]:
# Table at the largest N reached: who still finds the optimum?
if rows:
    N = max(r["N"] for r in rows)
    print(f"N = {N}")
    print(f"{'problem':10s} {'method':16s} {'regime':20s} {'p_success':>9s} {'residual':>9s}")
    for r in sorted((r for r in rows if r["N"] == N), key=lambda r: (r["problem"], r["kind"], r["method"])):
        print(f"{r['problem']:10s} {r['method']:16s} {r['regime']:20s} {r['p_success_median']:9.2f} {r['residual_median']:9.4f}")

In [ ]:
# Zephyr vs Pegasus (and any other pair of back-ends of the same device)
comp = ms.compare_backends(rows) if rows else []
for c in comp:
    print(f"{c['device']} {c['problem']:10s} N={c['N']:4d} winner: {c['winner']}")
# QPU bookkeeping: access time actually spent, chain breaks, chain length
for r in rows:
    if r["kind"] == "anneal" and r["method"].startswith("anneal_qpu"):
        print(f"{r['method']:28s} {r['problem']:10s} N={r['N']:4d} qpu_access={r['qpu_access_s']:.3f}s "
              f"chain_breaks={r['chain_break_fraction']} chain_len={r.get('chain_len_mean')}")

## Reading the results

* **`eps0_J03` (ferro_easy)**: every method, the annealer and the game should stay at $p\approx1$ as $N$ grows.
* **`eps0_J00` / `eps1_J00` (spin glass)**: $p$ of the annealer decays with $N$ (TTS$_{99}$ grows); the game freezes on fixed points with residual $\approx 5$–$10\%$ (*satisficing*), never on the optimum at large $N$; at $\alpha=1$ (cycles), $\varepsilon=1$ (chaos) and $T>T_c$ (paramagnet) it does not even get close.
* **`eps0_J01`**: the boundary, in between.
* Above $n_{\rm exact}$ the reference is only the best known: `ref_from_annealer > 0` in `summary.csv` means the annealer improved on every classical solver.